# DSCI 552 — Homework 3

- **Name:** Danmu Agudanmu
- **GitHub Username:** uudam42

In [61]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.dpi"] = 100

## Problem 1: Time Series Classification Part 1 — Feature Creation/Extraction

### 1(a) Loading the AReM data

The AReM data is saved in `data/AReM/`. It has 7 activity folders, and each file is one instance with 6 time series.

Some files need cleaning, so I read them line by line:
- Lines starting with `#` are headers and are skipped.
- `bending2/dataset4.csv` uses spaces instead of commas.
- Some rows end with an extra comma, so only the first 7 values of each row are kept.

In [63]:
DATA_DIR = Path("../data/AReM")
COLUMNS = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23"]
SERIES = COLUMNS[1:]

def read_arem(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            vals = [v for v in re.split(r"[,\s]+", line) if v]
            rows.append([float(v) for v in vals[:7]])
    return pd.DataFrame(rows, columns=COLUMNS)

def file_number(path):
    return int(re.search(r"\d+", path.stem).group())

instances = []
for folder in sorted(p for p in DATA_DIR.iterdir() if p.is_dir()):
    for f in sorted(folder.glob("dataset*.csv"), key=file_number):
        instances.append({
            "activity": folder.name,
            "dataset": file_number(f),
            "data": read_arem(f),
        })

print(f"Number of instances: {len(instances)}")
check = pd.DataFrame({
    "activity": [d["activity"] for d in instances],
    "dataset": [d["dataset"] for d in instances],
    "rows": [len(d["data"]) for d in instances],
    "missing values": [int(d["data"].isna().sum().sum()) for d in instances],
})
print("Files that do not have exactly 480 rows:")
check[check["rows"] != 480]

Number of instances: 88
Files that do not have exactly 480 rows:


,activity,dataset,rows,missing values
50,sitting,8,479,0


All 88 instances load with no missing values. `sitting/dataset8.csv` has 479 rows instead of 480. One missing time step does not really change the statistics, so I kept it.

### 1(b) Train/test split

Test set: datasets 1–2 in `bending1` and `bending2`, and datasets 1–3 in the other folders. The rest is the training set.

In [66]:
def is_test(activity, dataset):
    if activity.startswith("bending"):
        return dataset <= 2
    return dataset <= 3

for d in instances:
    d["split"] = "test" if is_test(d["activity"], d["dataset"]) else "train"

split_table = (pd.DataFrame([(d["activity"], d["split"]) for d in instances], columns=["activity", "split"])
               .value_counts().unstack(fill_value=0)[["train", "test"]])
split_table.loc["Total"] = split_table.sum()
split_table

split,train,test
activity,,
bending1,5,2
bending2,4,2
cycling,12,3
lying,12,3
sitting,12,3
standing,12,3
walking,12,3
Total,69,19


### 1(c) Feature extraction

#### 1(c)i. Common time-domain features

- Location: mean, median, mode
- Spread: standard deviation, variance, range, interquartile range, mean absolute deviation
- Order statistics: minimum, maximum, 1st quartile, 3rd quartile, other percentiles
- Shape: skewness, kurtosis
- Energy: sum of squares, root mean square (RMS)
- Change over time: mean absolute difference between consecutive values, number of peaks, number of mean crossings, slope
- Correlation: autocorrelation, correlation between two series

#### 1(c)ii. Extracting min, max, mean, median, std, 1st quartile and 3rd quartile

7 features for each of the 6 series gives 42 features per instance. Columns are named `<feature>_<series>`, with series 1–6 = `avg_rss12`, `var_rss12`, `avg_rss13`, `var_rss13`, `avg_rss23`, `var_rss23`. The features are not normalized.

In [68]:
FEATURES = ["min", "max", "mean", "median", "std", "1st_quart", "3rd_quart"]

def extract_features(ts):
    """ts: DataFrame with the 6 series as columns. Returns a dict of 42 features."""
    out = {}
    for i, s in enumerate(SERIES, start=1):
        x = ts[s].to_numpy()
        out[f"min_{i}"] = x.min()
        out[f"max_{i}"] = x.max()
        out[f"mean_{i}"] = x.mean()
        out[f"median_{i}"] = np.median(x)
        out[f"std_{i}"] = x.std(ddof=1)
        out[f"1st_quart_{i}"] = np.percentile(x, 25)
        out[f"3rd_quart_{i}"] = np.percentile(x, 75)
    return out

rows = []
for d in instances:
    rows.append({"activity": d["activity"], "dataset": d["dataset"], "split": d["split"],
                 **extract_features(d["data"])})

feat_df = pd.DataFrame(rows)
feat_df.index = pd.RangeIndex(1, len(feat_df) + 1, name="Instance")

# order columns as in the homework table: all features of series 1, then series 2, ...
feature_cols = [f"{f}_{i}" for i in range(1, 7) for f in FEATURES]
feat_df = feat_df[["activity", "dataset", "split"] + feature_cols]
print(feat_df.shape)
with pd.option_context("display.max_rows", 100):
    display(feat_df)

(88, 45)


,activity,dataset,split,min_1,max_1,mean_1,median_1,std_1,1st_quart_1,3rd_quart_1,min_2,max_2,mean_2,median_2,std_2,1st_quart_2,3rd_quart_2,min_3,max_3,mean_3,median_3,std_3,1st_quart_3,3rd_quart_3,min_4,max_4,mean_4,median_4,std_4,1st_quart_4,3rd_quart_4,min_5,max_5,mean_5,median_5,std_5,1st_quart_5,3rd_quart_5,min_6,max_6,mean_6,median_6,std_6,1st_quart_6,3rd_quart_6
Instance,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,bending1,1,test,37.2500,45.0000,40.6248,40.5000,1.4770,39.2500,42.0000,0.0000,1.3000,0.3586,0.4300,0.3226,0.0000,0.5000,4.0000,29.5000,19.0409,19.2500,4.4630,16.0000,23.2500,0.0000,7.2300,0.8325,0.5000,0.9657,0.0000,1.1200,27.2500,38.2500,34.3113,35.0000,2.1884,33.0000,36.0000,0.0000,1.9200,0.5706,0.4300,0.5829,0.0000,1.3000
2,bending1,2,test,38.0000,45.6700,42.8128,42.5000,1.4355,42.0000,43.6700,0.0000,1.2200,0.3724,0.4700,0.2892,0.0000,0.5000,2.0000,29.5000,20.0970,21.0000,3.8937,19.0000,22.2500,0.0000,5.7600,0.8764,0.5000,1.0473,0.0000,1.1450,27.6700,38.5000,33.0246,33.0000,1.9953,32.0000,34.5000,0.0000,3.1100,0.5711,0.4300,0.6010,0.0000,1.3000
3,bending1,3,train,35.0000,47.4000,43.9545,44.3300,1.5588,43.0000,45.0000,0.0000,1.7000,0.4263,0.4700,0.3387,0.0000,0.5000,6.5000,29.7500,22.1224,23.0000,3.0309,19.7500,24.0000,0.0000,4.4400,0.4973,0.4300,0.5507,0.0000,0.8300,29.0000,38.5000,35.5885,36.0000,1.9996,35.3625,36.5000,0.0000,1.7900,0.4933,0.4300,0.5135,0.0000,0.9400
4,bending1,4,train,33.0000,47.7500,42.1798,43.5000,3.6707,39.1500,45.0000,0.0000,3.0000,0.6960,0.5000,0.6309,0.0000,1.1200,8.5000,30.0000,22.1836,23.0000,3.8105,20.5000,24.3725,0.0000,5.1500,0.9899,0.8300,0.9537,0.4300,1.3000,20.0000,38.6700,33.4939,35.0000,3.8494,30.4575,36.3300,0.0000,2.1800,0.6135,0.5000,0.5243,0.0000,1.0000
5,bending1,5,train,33.0000,45.7500,41.6781,41.7500,2.2435,41.3300,42.7500,0.0000,2.8300,0.5360,0.5000,0.4055,0.4300,0.7100,3.0000,28.2500,19.0066,19.1250,4.0871,16.5000,22.0625,0.0000,6.4200,0.8419,0.5000,0.9288,0.4300,1.1200,23.6700,37.5000,29.8571,30.0000,2.4110,28.4575,31.2500,0.0000,1.7900,0.3833,0.4300,0.3892,0.0000,0.5000
6,bending1,6,train,37.0000,48.0000,43.4550,43.2500,1.3861,42.5000,45.0000,0.0000,1.5800,0.3781,0.4700,0.3156,0.0000,0.5000,5.7500,27.0000,15.7933,15.0000,3.8476,13.0000,18.2700,0.0000,10.0300,0.8494,0.5000,0.9958,0.4300,1.1200,8.0000,33.5000,23.0348,23.5000,2.4889,22.2500,24.0000,0.0000,5.2600,0.6796,0.5000,0.6225,0.4300,0.8700
7,bending1,7,train,36.2500,48.0000,43.9691,44.5000,1.6184,43.3100,44.6700,0.0000,1.5000,0.4131,0.4700,0.2631,0.4300,0.5000,1.5000,26.3300,15.8680,16.2500,3.7424,14.2500,18.0000,0.0000,5.1700,0.6664,0.4700,0.7890,0.0000,0.9400,11.3300,30.7500,22.1038,21.6700,3.3183,20.5000,23.7500,0.0000,2.9600,0.5553,0.4900,0.4878,0.0000,0.8300
8,bending2,1,test,12.7500,51.0000,24.5630,24.2500,3.7375,23.1875,26.5000,0.0000,6.8700,0.5908,0.4300,0.8374,0.0000,0.7100,0.0000,25.3300,19.1213,20.2500,4.2996,18.0000,22.0000,0.0000,6.7600,0.7368,0.4700,0.9270,0.0000,0.8700,10.6700,30.0000,23.4930,23.7500,3.6938,20.5000,27.0000,0.0000,4.9700,0.7002,0.5000,0.6937,0.4300,0.8700
9,bending2,2,test,0.0000,42.7500,27.4646,28.0000,3.5836,25.5000,30.0000,0.0000,7.7600,0.4497,0.4300,0.7672,0.0000,0.5000,7.5000,35.0000,20.8425,20.7500,3.8263,18.0000,22.5000,0.0000,5.7600,0.7799,0.5000,0.9044,0.0000,0.9400,0.0000,33.0000,17.6179,18.0000,5.0536,15.0000,20.7500,0.0000,6.7600,1.1221,0.8300,1.0123,0.4700,1.3000


In [69]:
train_df = feat_df[feat_df["split"] == "train"]
test_df = feat_df[feat_df["split"] == "test"]
print(f"Train instances: {len(train_df)}, Test instances: {len(test_df)}")

# save for Homework 4
feat_df.to_csv("../data/arem_features.csv")

Train instances: 69, Test instances: 19


#### 1(c)iii. Standard deviation of each feature with a 90% bootstrap confidence interval

I resample the 88 instances with replacement 2000 times, compute the standard deviation of each feature every time, and use the 5th and 95th percentiles as the 90% CI.

In [71]:
rng = np.random.default_rng(552)
B = 2000
X = feat_df[feature_cols].to_numpy()
n = X.shape[0]

boot_std = np.empty((B, X.shape[1]))
for b in range(B):
    idx = rng.integers(0, n, size=n)
    boot_std[b] = X[idx].std(axis=0, ddof=1)

std_table = pd.DataFrame({
    "Std. dev.": X.std(axis=0, ddof=1),
    "90% CI lower": np.percentile(boot_std, 5, axis=0),
    "90% CI upper": np.percentile(boot_std, 95, axis=0),
}, index=feature_cols)
std_table["CI width"] = std_table["90% CI upper"] - std_table["90% CI lower"]
std_table

,Std. dev.,90% CI lower,90% CI upper,CI width
min_1,9.5700,8.2529,10.7615,2.5086
max_1,4.3944,3.3631,5.2696,1.9065
mean_1,5.3357,4.6948,5.9134,1.2186
median_1,5.4401,4.7785,6.0125,1.2340
std_1,1.7722,1.5647,1.9468,0.3821
1st_quart_1,6.1536,5.5490,6.6770,1.1280
3rd_quart_1,5.1389,4.3561,5.8383,1.4822
min_2,0.0000,0.0000,0.0000,0.0000
max_2,5.0627,4.6387,5.3949,0.7562
mean_2,1.5742,1.3914,1.7027,0.3113


- `min_2` and `min_4` have a standard deviation of 0. The minimum of `var_rss12` and `var_rss13` is 0 in all 88 instances, so these two features are useless.
- The `avg_rss` series (1, 3, 5) are on a larger scale than the `var_rss` series (2, 4, 6), so their features have larger standard deviations.
- `min_5` and `min_1` have the widest intervals.

#### 1(c)iv. Selecting the three most important time-domain features

A useful feature should take different values for different activities. On the training set, I computed the ANOVA F-statistic of each feature across the activities (bending1 and bending2 combined), then averaged it over the 6 series. `min_2` and `min_4` are 0 in every instance, so their F is left as NaN.

In [74]:
y_train = train_df["activity"].replace({"bending1": "bending", "bending2": "bending"})

def anova_f(col):
    if train_df[col].nunique() == 1:   # min_2 and min_4 are always 0, so F is undefined
        return np.nan
    groups = [train_df.loc[y_train == a, col] for a in y_train.unique()]
    return stats.f_oneway(*groups).statistic

F = pd.Series({c: anova_f(c) for c in feature_cols})
F_table = pd.DataFrame(
    {f"series {i} ({SERIES[i-1]})": [F[f"{f}_{i}"] for f in FEATURES] for i in range(1, 7)},
    index=FEATURES)
F_table["mean F"] = F_table.mean(axis=1)
F_table.sort_values("mean F", ascending=False)

,series 1 (avg_rss12),series 2 (var_rss12),series 3 (avg_rss13),series 4 (var_rss13),series 5 (avg_rss23),series 6 (var_rss23),mean F
mean,8.8670,"1,089.9550",22.3709,794.4435,19.9633,540.6630,412.7104
3rd_quart,5.0189,979.3307,28.0718,684.1371,21.0337,716.1229,405.6192
median,8.4851,570.5738,17.2072,969.3605,15.9824,575.0245,359.4389
1st_quart,10.9545,314.3802,16.8512,290.7155,14.7350,324.9548,162.0985
std,7.7448,415.1665,12.4025,155.8904,4.7598,121.8865,119.6418
max,1.5354,51.7635,37.0921,34.8536,43.3650,21.3260,31.6559
min,10.5978,NaN,20.1067,NaN,11.5474,0.9462,10.7995


The F-statistic only looks at one feature at a time. As a second check, I compared all 35 combinations of 3 feature types with 5-fold cross-validation (repeated 5 times) on the training set, using logistic regression on standardized features.

In [76]:
import itertools
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=552)
rows = []
for combo in itertools.combinations(FEATURES, 3):
    cols = [f"{f}_{i}" for f in combo for i in range(1, 7)]
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
    acc = cross_val_score(model, train_df[cols], y_train, cv=cv)
    rows.append({"features": ", ".join(combo), "CV accuracy": acc.mean(), "CV std": acc.std()})
combo_table = pd.DataFrame(rows).sort_values("CV accuracy", ascending=False).reset_index(drop=True)
combo_table.index += 1
combo_table.head(10)

,features,CV accuracy,CV std
1,"max, mean, 3rd_quart",0.9160,0.0723
2,"min, max, 3rd_quart",0.9099,0.0751
3,"max, 1st_quart, 3rd_quart",0.8980,0.0945
4,"max, std, 3rd_quart",0.8958,0.0762
5,"max, median, 3rd_quart",0.8921,0.0701
6,"max, mean, std",0.8901,0.0677
7,"min, mean, std",0.8842,0.0841
8,"max, median, std",0.8842,0.0578
9,"mean, median, std",0.8811,0.0679
10,"min, std, 3rd_quart",0.8785,0.0829


In [77]:
print(combo_table[combo_table["features"] == "min, max, mean"])
print()
print("Correlation between mean and median of the same series (training set):")
print(pd.Series({i: train_df[f"mean_{i}"].corr(train_df[f"median_{i}"]) for i in range(1, 7)}).round(3).to_string())

          features  CV accuracy  CV std
13  min, max, mean       0.8723  0.0878

Correlation between mean and median of the same series (training set):
1   0.9740
2   0.9940
3   0.9910
4   0.9950
5   0.9930
6   0.9960


I chose **mean, max and 3rd quartile**.

- Mean has the highest average F-statistic and is in the top 3 for every series.
- 3rd quartile has the second highest F-statistic.
- Max is weak on its own, but it is the best single feature for series 3 and 5. It also adds something the other two don't capture.

This combination also had the highest CV accuracy (0.916). Median has a high F-statistic too, but its correlation with mean is above 0.97 for every series, so it adds almost nothing. Min is 0 for series 2 and 4 in every instance, so it is a weak choice here. Min, mean and max scored 0.872.

## Problem 2: ISLR 3.7.4

n = 100 observations, one predictor. We fit a linear regression and a cubic regression ($Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \beta_3 X^3 + \varepsilon$).

**(a)** True relationship is linear, training RSS: the cubic model will have lower training RSS. The linear model is a special case of the cubic model ($\beta_2 = \beta_3 = 0$), so least squares can always fit the training data at least as well with the cubic model. The extra terms will also fit some of the noise.

**(b)** True relationship is linear, test RSS: the linear model should have lower test RSS. The true relationship is linear, so the extra cubic terms do not reduce bias. They only add variance by fitting noise, which makes test error worse.

**(c)** True relationship is not linear, training RSS: the cubic model will have lower training RSS, for the same reason as in (a). This holds no matter what the true relationship is.

**(d)** True relationship is not linear, test RSS: there is not enough information to tell. If the true relationship is far from linear, the cubic model should have lower test RSS because the linear model has high bias. If it is close to linear, the linear model may still do better because the cubic model has higher variance.